# DAF-Net: Deepfake Artifact-Focusing Network
### Kaggle GPU Experiment Runner (V1 to V5 & Baselines)

- **Dataset**: `manjilkarki/deepfake-and-real-images` (~140k Train, ~39k Val, ~11k Test)
- **Fixed Protocol**: AdamW (lr=1e-4, wd=1e-4), Cosine Annealing, BCEWithLogitsLoss, Batch Size 64, Mixed Precision (AMP)
- **Resolution**: 128x128 for rapid exploration across V1-V5
- **Architecture**: 3 Progressive Artifact Blocks (|3x3 - 5x5| deviation -> Gating -> Residual Amplification)

In [ ]:
import os
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Accelerator: {torch.cuda.get_device_name(0)}')


## 1. Locate Dataset on Kaggle
Checks Kaggle input directory for the `deepfake-and-real-images` dataset.

In [ ]:
DATA_ROOT = None
candidates = [
    '/kaggle/input/deepfake-and-real-images',
    '/kaggle/input/deepfake-and-real-images/deepfake_and_real_images',
    '/kaggle/input/deepfake-and-real-images/dataset-140k/dataset-140k'
]
for p in candidates:
    if os.path.exists(p):
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if any('real' in d.lower() for d in dirs) and any('fake' in d.lower() for d in dirs):
            DATA_ROOT = root
            break

print('Dataset Root:', DATA_ROOT)
if DATA_ROOT:
    print('Subdirectories:', os.listdir(DATA_ROOT))


## 2. Dataset Loader with Class Balance Verification

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir: str, transform=None, max_samples=None):
        self.samples = []
        self.class_counts = {0: 0, 1: 0}  # 0=Real, 1=Fake
        self.transform = transform
        
        print(f'Scanning dataset directory: {root_dir} ...')
        for root, dirs, files in os.walk(root_dir):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    parent = os.path.basename(root).lower()
                    if 'real' in parent:
                        self.samples.append((os.path.join(root, f), 0))
                        self.class_counts[0] += 1
                    elif 'fake' in parent:
                        self.samples.append((os.path.join(root, f), 1))
                        self.class_counts[1] += 1
                    if max_samples and len(self.samples) >= max_samples:
                        break
            if max_samples and len(self.samples) >= max_samples:
                break
        
        print(f'-> Done! Loaded {len(self.samples)} images from {root_dir}')
        print(f'   Class Balance -> Real: {self.class_counts[0]} | Fake: {self.class_counts[1]}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)


## 3. DAF-Net Architectural Blocks (V1, V2, V3, V4 and Baseline)

In [ ]:
class DownsampleBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.1, inplace=True)
        )
    def forward(self, x):
        return self.block(x)

# --- V1: Basic 3x3 vs 5x5 + Learnable alpha ---
class ArtifactBlockV1(nn.Module):
    def __init__(self, c, init_alpha=1.0):
        super().__init__()
        self.local_branch = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True)
        )
        self.context_branch = nn.Sequential(
            nn.Conv2d(c, c, 5, padding=2, bias=False),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True)
        )
        self.gate_conv = nn.Sequential(
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 1, bias=True),
            nn.Sigmoid()
        )
        self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

    def forward(self, x, return_artifacts=False):
        f_loc = self.local_branch(x)
        f_ctx = self.context_branch(x)
        dev = torch.abs(f_loc - f_ctx)
        gate = self.gate_conv(dev)
        out = x + self.alpha * (x * gate)
        if return_artifacts:
            return out, {'deviation': dev, 'gate': gate, 'alpha': self.alpha.item()}
        return out

# --- V2: Refined Depthwise-Separable & Dual Spatial-Channel Gating ---
class ArtifactBlockV2(nn.Module):
    def __init__(self, c, init_alpha=1.0):
        super().__init__()
        self.local_branch = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, groups=c, bias=False),
            nn.Conv2d(c, c, 1, bias=False),
            nn.BatchNorm2d(c), nn.ReLU(inplace=True)
        )
        self.context_branch = nn.Sequential(
            nn.Conv2d(c, c, 5, padding=2, groups=c, bias=False),
            nn.Conv2d(c, c, 1, bias=False),
            nn.BatchNorm2d(c), nn.ReLU(inplace=True)
        )
        self.spatial_gate = nn.Sequential(nn.Conv2d(c, 1, 1), nn.Sigmoid())
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Conv2d(c, max(c // 4, 8), 1), nn.ReLU(inplace=True),
            nn.Conv2d(max(c // 4, 8), c, 1), nn.Sigmoid()
        )
        self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

    def forward(self, x, return_artifacts=False):
        dev = torch.abs(self.local_branch(x) - self.context_branch(x))
        gate = self.spatial_gate(dev) * self.channel_gate(dev)
        out = x + self.alpha * (x * gate)
        if return_artifacts:
            return out, {'deviation': dev, 'gate': gate, 'alpha': self.alpha.item()}
        return out

# --- V3: Frequency-Aware Gating (Spatial + Laplacian High-Pass) ---
class ArtifactBlockV3(nn.Module):
    def __init__(self, c, init_alpha=1.0):
        super().__init__()
        self.c = c
        self.local_branch = nn.Sequential(nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.context_branch = nn.Sequential(nn.Conv2d(c, c, 5, padding=2, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        lap = torch.tensor([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]], dtype=torch.float32).view(1, 1, 3, 3).repeat(c, 1, 1, 1)
        self.register_buffer('lap', lap)
        self.freq_fusion = nn.Sequential(nn.Conv2d(c * 2, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.gate_conv = nn.Sequential(nn.Conv2d(c, c, 1), nn.Sigmoid())
        self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

    def forward(self, x, return_artifacts=False):
        d_sp = torch.abs(self.local_branch(x) - self.context_branch(x))
        d_fq = torch.abs(F.conv2d(x, self.lap, padding=1, groups=self.c))
        fused = self.freq_fusion(torch.cat([d_sp, d_fq], dim=1))
        gate = self.gate_conv(fused)
        out = x + self.alpha * (x * gate)
        if return_artifacts:
            return out, {'deviation': fused, 'gate': gate, 'alpha': self.alpha.item()}
        return out

# --- V4: Adaptive Alpha(x) Amplification ---
class ArtifactBlockV4(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.local_branch = nn.Sequential(nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.context_branch = nn.Sequential(nn.Conv2d(c, c, 5, padding=2, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.gate_conv = nn.Sequential(nn.BatchNorm2d(c), nn.ReLU(inplace=True), nn.Conv2d(c, c, 1), nn.Sigmoid())
        self.adaptive_alpha = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Conv2d(c, max(c // 4, 8), 1), nn.ReLU(inplace=True),
            nn.Conv2d(max(c // 4, 8), 1, 1), nn.Sigmoid()
        )
    def forward(self, x, return_artifacts=False):
        dev = torch.abs(self.local_branch(x) - self.context_branch(x))
        gate = self.gate_conv(dev)
        alpha = self.adaptive_alpha(x) * 2.0
        out = x + alpha * (x * gate)
        if return_artifacts:
            return out, {'deviation': dev, 'gate': gate, 'alpha': alpha.squeeze().mean().item()}
        return out

# --- Full Parametric DAF-Net ---
class DAFNet(nn.Module):
    def __init__(self, version='v1', in_c=3, stem_c=32, stage_c=(64, 128, 256), dropout=0.2, init_alpha=1.0):
        super().__init__()
        self.version = version.lower()
        c1, c2, c3 = stage_c
        self.stem = nn.Sequential(
            nn.Conv2d(in_c, stem_c, 3, 1, 1, bias=False), nn.BatchNorm2d(stem_c), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(stem_c, c1, 3, 1, 1, bias=False), nn.BatchNorm2d(c1), nn.LeakyReLU(0.1, inplace=True)
        )
        if self.version == 'v1':
            self.ab1, self.ab2, self.ab3 = ArtifactBlockV1(c1, init_alpha), ArtifactBlockV1(c2, init_alpha), ArtifactBlockV1(c3, init_alpha)
        elif self.version == 'v2':
            self.ab1, self.ab2, self.ab3 = ArtifactBlockV2(c1, init_alpha), ArtifactBlockV2(c2, init_alpha), ArtifactBlockV2(c3, init_alpha)
        elif self.version == 'v3':
            self.ab1, self.ab2, self.ab3 = ArtifactBlockV3(c1, init_alpha), ArtifactBlockV3(c2, init_alpha), ArtifactBlockV3(c3, init_alpha)
        elif self.version == 'v4':
            self.ab1, self.ab2, self.ab3 = ArtifactBlockV4(c1), ArtifactBlockV4(c2), ArtifactBlockV4(c3)
        
        self.ds1 = DownsampleBlock(c1, c2)
        self.ds2 = DownsampleBlock(c2, c3)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(c3, 1)

    def forward(self, x, return_artifacts=False):
        feat = self.stem(x)
        if return_artifacts:
            feat, art1 = self.ab1(feat, True)
            feat = self.ds1(feat)
            feat, art2 = self.ab2(feat, True)
            feat = self.ds2(feat)
            feat, art3 = self.ab3(feat, True)
            logit = self.fc(self.dropout(self.gap(feat).flatten(1)))
            return logit, {'block1': art1, 'block2': art2, 'block3': art3}
        else:
            feat = self.ab1(feat)
            feat = self.ds1(feat)
            feat = self.ab2(feat)
            feat = self.ds2(feat)
            feat = self.ab3(feat)
            return self.fc(self.dropout(self.gap(feat).flatten(1)))


## 4. Standardized Evaluation & Training Engine

In [ ]:
def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    y_true, y_prob = [], []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(imgs)
                loss = criterion(logits, labels)
                probs = torch.sigmoid(logits)
            total_loss += loss.item()
            y_true.extend(labels.cpu().squeeze().tolist())
            y_prob.extend(probs.cpu().squeeze().tolist())
            
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    
    return {
        'loss': total_loss / len(loader),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'cm': confusion_matrix(y_true, y_pred).tolist()
    }

def train_dafnet(model, train_loader, val_loader, epochs=12, lr=1e-4, wd=1e-4, model_name='dafnet_v1'):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    best_f1 = -1.0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': [], 'val_acc': []}
    total_batches = len(train_loader)
    
    print(f'=== Starting Training: {model_name} on {device} ({total_batches} batches/epoch) ===')
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        start_time = time.time()
        
        for step, (imgs, labels) in enumerate(train_loader, 1):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            
            if step % 100 == 0 or step == total_batches:
                elapsed = time.time() - start_time
                avg_b_loss = epoch_loss / step
                print(f'  [Epoch {epoch:02d} | Batch {step:04d}/{total_batches:04d}] Batch Loss: {avg_b_loss:.4f} ({elapsed:.1f}s)')
                
        print(f'Evaluating validation set for Epoch {epoch} ...')
        val_res = evaluate_model(model, val_loader, criterion, device)
        scheduler.step()
        
        train_loss = epoch_loss / total_batches
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_res['loss'])
        history['val_f1'].append(val_res['f1'])
        history['val_auc'].append(val_res['roc_auc'])
        history['val_acc'].append(val_res['accuracy'])
        
        print(f"\n>>> Epoch [{epoch:02d}/{epochs:02d}] Summary:")
        print(f"    Train Loss: {train_loss:.4f} | Val Loss: {val_res['loss']:.4f}")
        print(f"    Val Acc:    {val_res['accuracy']:.4f} | Val F1: {val_res['f1']:.4f} | AUC: {val_res['roc_auc']:.4f}\n")
        
        if val_res['f1'] > best_f1:
            best_f1 = val_res['f1']
            torch.save(model.state_dict(), f"{model_name}_best.pth")
            print(f"  --> [SAVED BEST CHECKPOINT] New highest F1: {best_f1:.4f}\n")
            
    with open(f"{model_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return history


## 5. Prepare DataLoaders (128x128)

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 64

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if DATA_ROOT is None:
    raise RuntimeError('ERROR: Dataset path not found! Make sure you clicked + Add Data and added manjilkarki/deepfake-and-real-images.')

train_dir = os.path.join(DATA_ROOT, 'train') if os.path.exists(os.path.join(DATA_ROOT, 'train')) else DATA_ROOT
val_dir = os.path.join(DATA_ROOT, 'valid') if os.path.exists(os.path.join(DATA_ROOT, 'valid')) else None
if val_dir is None and os.path.exists(os.path.join(DATA_ROOT, 'val')):
    val_dir = os.path.join(DATA_ROOT, 'val')

train_dataset = DeepfakeDataset(train_dir, transform=train_transform)
if val_dir and os.path.exists(val_dir):
    val_dataset = DeepfakeDataset(val_dir, transform=val_transform)
else:
    print('No separate validation folder found. Splitting train into 85% train / 15% val...')
    train_size = int(0.85 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Ready: {len(train_dataset)} train samples ({len(train_loader)} batches) | {len(val_dataset)} val samples ({len(val_loader)} batches)')


## 6. Train DAF-Net V1
Verifies working baseline and logs validation metrics.

In [ ]:
model_v1 = DAFNet(version='v1')
total_params = sum(p.numel() for p in model_v1.parameters())
print(f'DAF-Net V1 Total Parameters: {total_params:,}')

# Run training
history_v1 = train_dafnet(model_v1, train_loader, val_loader, epochs=12, lr=1e-4, model_name='dafnet_v1')


## 7. Visualize Artifact Maps
Visualizes the learned deviation and gating maps across Stage 1, 2, and 3.

In [ ]:
model_v1.eval()
sample_imgs, sample_labels = next(iter(val_loader))
sample_imgs = sample_imgs.to(device)

with torch.no_grad():
    logits, art_maps = model_v1(sample_imgs[:4], return_artifacts=True)
    probs = torch.sigmoid(logits).cpu().numpy().flatten()

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for row in range(4):
    img = sample_imgs[row].cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    gt = 'Fake' if sample_labels[row].item() == 1 else 'Real'
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'GT: {gt} | Pred: {probs[row]:.2f}')
    axes[row, 0].axis('off')
    
    for col, blk in enumerate(['block1', 'block2', 'block3'], start=1):
        gate = art_maps[blk]['gate'][row].mean(dim=0).cpu().numpy()
        axes[row, col].imshow(gate, cmap='viridis')
        axes[row, col].set_title(f"{blk.upper()} Gate")
        axes[row, col].axis('off')
        
plt.tight_layout()
plt.show()


## 8. Incremental Experiments: Train V2, V3, and V4
Uncomment each cell as you progress through the roadmap.

In [ ]:
# --- Train V2: Refined Depthwise & Dual Gating ---
# model_v2 = DAFNet(version='v2')
# history_v2 = train_dafnet(model_v2, train_loader, val_loader, epochs=12, lr=1e-4, model_name='dafnet_v2')

# --- Train V3: Frequency-Aware Gating ---
# model_v3 = DAFNet(version='v3')
# history_v3 = train_dafnet(model_v3, train_loader, val_loader, epochs=12, lr=1e-4, model_name="dafnet_v3")

# --- Train V4: Adaptive Alpha(x) ---
# model_v4 = DAFNet(version='v4')
# history_v4 = train_dafnet(model_v4, train_loader, val_loader, epochs=12, lr=1e-4, model_name="dafnet_v4")
